In [ ]:
import networkx as nx
import numpy as np
import functools

In [ ]:
@functools.lru_cache(maxsize=None)  # 没有缓存限制
def all_kshell_values(graph: nx.Graph) -> dict['Node' : 'k_shell_value']:
    
    return nx.core_number(graph)  

@functools.lru_cache(maxsize=None)  # 没有缓存限制
def node_shortest_path_from_node(graph, source) -> dict['Node' : 'distance']:
    '''
    计算源节点节点source到其他节点的最短路径，包括源节点本身
    返回值为dict[source node to target node, distance]
    '''
    
    return nx.shortest_path_length(graph, source)

In [ ]:
def neighbors_node_underneath_radius(graph: nx.Graph, i: 'node', radius: int) -> list:
    neighbors = []

    # 存储源节点i到其它节点的最短距离,除到源节点距离为0之外，无路径的节点对不会返回键值对。如{2: 0, 7: 1}
    i_shortest_path = node_shortest_path_from_node(graph, i)
    
    for node in i_shortest_path.keys():     
        if (node != i) & (i_shortest_path[node] <= radius):
            neighbors.append(node)

    # 孤立节点的neighbors为空列表
    return neighbors

In [ ]:
def gravity_centrality(graph, i) -> float:
    '''
    采用k_shell值作为质量，最短路径作为距离，截断半径根据网络特性自己设置，一般为网络直径的一半
    计算节点i的重力中心性。

    graph->所要计算的图
    node->所要计算重力中心性的节点
    所有节点的k_shell值，可用nx.core_number(graph)函数计算。返回值为dic{node : value}
    influence_radius->节点node的影响半径.
    '''
    
    influence_radius = 2
    gravitation = 0
    
    # 距离在influence_radius以内的邻居节点
    neighbor_under_2 = neighbors_node_underneath_radius(graph, i, influence_radius)  
    path_lengths_from_i = node_shortest_path_from_node(graph, i)  # 存储源节点i到其它节点的最短距离
    k_shell = all_kshell_values(graph)
            
    for j in neighbor_under_2:
        value = k_shell[i] * k_shell[j]
        distance = path_lengths[j]
        gravitation += value / (distance ** 2)

    return gravitation

def all_gravity_centrality(G):
    '''
    循环调用gravity_centrality计算所有节点重力中心性
    GC返回值为dic{nod : value of gravity centrality}
    '''
    
    GC = {node : 0 for node in G.nodes()}
    for node in G.nodes():
        GC[node] = gravity_centrality(G, node)

    return GC